In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import scipy.io as sio
print("scipy loaded fine")

In [ ]:
import scipy.io as sio

path = "1_20160518 (1).mat"
data = sio.loadmat(path)

for key in data.keys():
    if not key.startswith("__"):
        val = data[key]
        print(key, val.shape if hasattr(val, "shape") else type(val))

In [ ]:
import scipy.io as sio
import numpy as np

session_labels = {
    1: [1,2,3,0,2,0,0,1,0,1,2,1,1,1,2,3,2,2,3,3,0,3,0,3],
    2: [2,1,3,0,0,2,0,2,3,3,2,3,2,0,1,1,2,1,0,3,0,1,3,1],
    3: [1,2,2,1,3,3,3,1,1,2,1,0,2,3,3,0,2,3,0,0,2,0,1,0],
}

def load_subject_session(mat_path, session_num):
    data = sio.loadmat(mat_path)
    labels = session_labels[session_num]

    trials = []
    for trial_idx in range(1, 25):   # trials 1 to 24
        key = f"de_LDS{trial_idx}"
        feat = data[key]              # shape: (62, T, 5)
        label = labels[trial_idx - 1] # 0=neutral,1=sad,2=fear,3=happy
        trials.append({"features": feat, "label": label})
    return trials

# test it
trials = load_subject_session("1_20160518 (1).mat", session_num=1)
print(len(trials), "trials loaded")
print(trials[0]["features"].shape, "label:", trials[0]["label"])
print(trials[5]["features"].shape, "label:", trials[5]["label"])

In [ ]:
def windows_from_trial(trial):
    feat = trial["features"]  # (62, T, 5)
    label = trial["label"]
    T = feat.shape[1]
    samples = []
    for t in range(T):
        samples.append({"features": feat[:, t, :], "label": label})  # (62, 5)
    return samples

In [ ]:
all_samples = []
for tr in trials:
    all_samples.extend(windows_from_trial(tr))

print(len(all_samples), "total windowed samples from this subject/session")
print(all_samples[0]["features"].shape)

In [ ]:
import os
import scipy.io as sio

session_labels = {
    1: [1,2,3,0,2,0,0,1,0,1,2,1,1,1,2,3,2,2,3,3,0,3,0,3],
    2: [2,1,3,0,0,2,0,2,3,3,2,3,2,0,1,1,2,1,0,3,0,1,3,1],
    3: [1,2,2,1,3,3,3,1,1,2,1,0,2,3,3,0,2,3,0,0,2,0,1,0],
}

def load_subject_session(mat_path, session_num):
    data = sio.loadmat(mat_path)
    labels = session_labels[session_num]
    trials = []
    for trial_idx in range(1, 25):
        key = f"de_LDS{trial_idx}"
        feat = data[key]
        label = labels[trial_idx - 1]
        trials.append({"features": feat, "label": label})
    return trials

def windows_from_trial(trial):
    feat = trial["features"]
    label = trial["label"]
    T = feat.shape[1]
    samples = []
    for t in range(T):
        samples.append({"features": feat[:, t, :], "label": label})
    return samples

def load_all_data(base_path):
    all_data = []
    for session_num in [1, 2, 3]:
        session_folder = os.path.join(base_path, str(session_num))
        files = [f for f in os.listdir(session_folder) if f.endswith(".mat")]

        for fname in files:
            subject_id = fname.split("_")[0]
            fpath = os.path.join(session_folder, fname)

            trials = load_subject_session(fpath, session_num)
            for tr in trials:
                windows = windows_from_trial(tr)
                for w in windows:
                    w["subject"] = int(subject_id)
                    w["session"] = session_num
                    all_data.append(w)
        print(f"Session {session_num} done, files: {len(files)}")
    return all_data

base_path = "eeg_feature_smooth"
all_data = load_all_data(base_path)
print(len(all_data), "total samples across all subjects and sessions")

In [ ]:
import time
start = time.time()
data = sio.loadmat("eeg_feature_smooth/1/1_20160518.mat")
print("took", time.time() - start, "seconds")

In [ ]:
import numpy as np

data = np.load("seed_iv_processed.npz")
print("Keys:", list(data.keys()))

features = data["features"]
labels = data["labels"]

print("Features shape:", features.shape)
print("Labels shape:", labels.shape)
print("Unique labels:", np.unique(labels, return_counts=True))

if "subject_ids" in data:
    print("Unique subjects:", np.unique(data["subject_ids"]))

In [ ]:
import numpy as np

# Load the processed dataset
dataset = np.load("seed_iv_processed.npz")

# Extract the arrays
features = dataset["features"]         # Shape: (37575, 62, 5)
labels = dataset["labels"]             # Shape: (37575,)
subject_ids = dataset["subject_ids"]   # Shape: (37575,)
session_nums = dataset["session_nums"] # Shape: (37575,)
trial_ids = dataset["trial_ids"]       # Shape: (37575,)

print("Loaded features shape:", features.shape)